In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\2025_Sirifort, Delhi - CPCB.xlsx",skiprows=16)

In [3]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,CO,Ozone,Benzene,Toluene,Eth-Benzene,RH,WS,WD,SR,BP,AT,RF,TOT-RF
0,01-01-2025 00:00,02-01-2025 00:00,219.70,309.52,22.95,28.55,51.45,1.04,22.06,0.27,NaN,0.03,87.11,1.49,154.72,91.52,NaN,10.87,0.0,0.0
1,02-01-2025 00:00,03-01-2025 00:00,219.89,321.44,24.53,29.35,53.91,1.13,21.63,0.43,NaN,0.24,88.80,1.73,155.75,85.88,NaN,10.96,0.0,0.0
2,03-01-2025 00:00,04-01-2025 00:00,299.41,456.70,25.46,24.12,49.55,2.31,21.59,0.23,NaN,0.02,88.50,0.75,113.66,84.88,NaN,12.44,0.0,0.0
3,04-01-2025 00:00,05-01-2025 00:00,316.88,401.50,22.10,32.27,54.36,2.78,20.39,0.25,NaN,0.04,91.40,0.69,85.49,83.17,NaN,12.27,0.0,0.0
4,05-01-2025 00:00,06-01-2025 00:00,217.47,296.21,23.49,32.96,56.45,1.07,20.28,0.26,NaN,0.03,90.33,0.98,97.74,84.32,NaN,11.55,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,12-11-2025 00:00,13-11-2025 00:00,171.92,503.43,22.86,39.34,39.51,1.96,10.44,1.33,NaN,0.40,67.99,1.98,167.77,96.05,NaN,18.27,0.0,0.0
316,13-11-2025 00:00,14-11-2025 00:00,304.11,436.53,23.55,41.20,41.06,1.93,10.51,1.23,NaN,0.35,68.83,1.09,164.30,96.68,NaN,18.09,0.0,0.0
317,14-11-2025 00:00,15-11-2025 00:00,232.26,384.53,24.52,40.89,41.68,1.32,11.46,1.25,NaN,0.42,68.96,1.05,155.77,92.80,NaN,17.62,0.0,0.0
318,15-11-2025 00:00,16-11-2025 00:00,246.41,428.80,23.72,38.71,39.88,1.49,9.81,1.37,NaN,0.40,68.69,1.04,162.82,91.89,NaN,17.57,0.0,0.0


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (320, 19)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['Toluene']
Dropped rows (>70% NaN): 0
Missing values after imputation:
 From Date      0
To Date        0
PM2.5          0
PM10           0
NO             0
NO2            0
NOx            0
CO             0
Ozone          0
Benzene        0
Eth-Benzene    0
RH             0
WS             0
WD             0
SR             0
AT             0
RF             0
TOT-RF         0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:

# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (320, 18)
          From Date           To Date    PM2.5    PM10     NO    NO2    NOx  \
0  01-01-2025 00:00  02-01-2025 00:00  219.700  309.52  22.95  28.55  51.45   
1  02-01-2025 00:00  03-01-2025 00:00   66.575  321.44  24.53  29.35  53.91   
2  03-01-2025 00:00  04-01-2025 00:00   66.575  456.70  25.46  36.21  49.55   
3  04-01-2025 00:00  05-01-2025 00:00   66.575  401.50  22.10  32.27  54.36   
4  05-01-2025 00:00  06-01-2025 00:00  217.470  296.21  23.49  32.96  56.45   

     CO  Ozone  Benzene  Eth-Benzene     RH    WS      WD     SR     AT   RF  \
0  1.04  22.06     1.32         0.44  87.11  1.49  154.72  91.52  10.87  0.0   
1  1.13  21.63     1.32         0.44  88.80  1.73  155.75  85.88  10.96  0.0   
2  0.80  21.59     1.32         0.44  88.50  0.75  113.66  84.88  12.44  0.0   
3  0.80  20.39     1.32         0.44  91.40  0.69   85.49  83.17  12.27  0.0   
4  1.07  20.28     1.32         0.44  90.33  0.98   97.74  84.32  11.55  0.0   

   TOT-RF  
0     0.0

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,CO,Ozone,Benzene,Eth-Benzene,RH,WS,WD,SR,AT,RF,TOT-RF
0,01-01-2025 00:00,02-01-2025 00:00,3.096307,1.300125,-0.993859,-2.519632,1.739620,0.349373,-0.528049,-0.034301,0.170231,1.071096,0.515266,0.382194,-0.996894,-2.404828,0.0,0.0
1,02-01-2025 00:00,03-01-2025 00:00,-0.163450,1.428186,-0.076763,-2.264306,2.052379,0.543619,-0.578632,-0.034301,0.170231,1.192295,0.903307,0.410000,-1.079771,-2.389983,0.0,0.0
2,03-01-2025 00:00,04-01-2025 00:00,-0.163450,2.881329,0.463046,-0.074892,1.498059,-0.168616,-0.583338,-0.034301,0.170231,1.170780,-0.681194,-0.726268,-1.094466,-2.145871,0.0,0.0
3,04-01-2025 00:00,05-01-2025 00:00,-0.163450,2.288297,-1.487234,-1.332369,2.109591,-0.168616,-0.724501,-0.034301,0.170231,1.378754,-0.778204,-1.486750,-1.119594,-2.173911,0.0,0.0
4,05-01-2025 00:00,06-01-2025 00:00,3.048834,1.157131,-0.680421,-1.112151,2.375308,0.414122,-0.737441,-0.034301,0.170231,1.302019,-0.309321,-1.156047,-1.102695,-2.292668,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,12-11-2025 00:00,13-11-2025 00:00,2.079157,-0.059122,-1.046099,0.924068,0.221597,2.334999,-1.894981,0.142736,-0.872331,-0.300093,1.307517,0.734494,-0.930327,-1.184268,0.0,0.0
316,13-11-2025 00:00,14-11-2025 00:00,-0.163450,2.664636,-0.645595,1.517699,0.418660,2.270251,-1.886747,-1.627628,-2.175532,-0.239853,-0.131469,0.640818,-0.921070,-1.213957,0.0,0.0
317,14-11-2025 00:00,15-11-2025 00:00,-0.163450,2.105983,-0.082568,1.418761,0.497486,0.953694,-1.774993,-1.273555,-0.351050,-0.230530,-0.196143,0.410540,-0.978085,-1.291479,0.0,0.0
318,15-11-2025 00:00,16-11-2025 00:00,-0.163450,2.581590,-0.546920,0.722999,0.268638,1.320603,-1.969092,0.850881,-0.872331,-0.249893,-0.212311,0.600863,-0.991457,-1.299726,0.0,0.0


In [10]:
df.to_excel('Sirifort2025.xlsx', index=False)